# VanillaNet-7 for ship classification: competition notebook

Author: Clément Castiglione

This is the notebook submitted for the competition, translated from French, with its original outputs, including the log of the final 300-epoch training run. The "Train Acc" in that log is computed on mixup/cutmix images, so it is much lower than the actual training accuracy.

The code maintained in this repository is in `src/vanillanet_ships/` (see the README). It differs from this notebook in a few places, for example the convolutions followed by a batch norm no longer have a bias.

We chose to adapt VanillaNet (https://arxiv.org/abs/2305.12972) to the competition.

This model is interesting because, with only a few layers, it competes with much deeper models.

It does so thanks to:
- A special activation function that increases the non-linearity of the model, the Series Informed Activation Function. For each input and its neighbourhood, a bias is added and a standard activation function is applied, then the inputs are summed with learned weights. This strongly breaks linearity, which lets the model learn more complex patterns with fewer layers.
- A strategy to merge two convolution layers into one after training. An activation function is placed between two convolutions and is gradually attenuated during training until it becomes the identity. After training, the two layers can then easily be merged into one. The batch norms after each convolution are not mentioned here because they can be folded into their convolution after training, by scaling the convolution weights with the parameters learned by the batch norm.

In [ ]:
import gc
import multiprocessing
import os

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

First, to make training simpler, we convert the dataset to numpy arrays and save them for reuse.

In [ ]:
def create_npz_from_images(imgs_path: str, dest_path: str):
    images = []
    labels = []

    labels_folders = sorted(os.listdir(imgs_path))

    labels_mapping = {e:i for i,e in enumerate(labels_folders)}

    for label, folder in enumerate(labels_folders):
        folder_path = os.path.join(imgs_path, folder)
        for file in os.listdir(folder_path):
            file_path = os.path.join(folder_path, file)
            im = np.array(Image.open(file_path))
            images.append(im)
            labels.append(label)

    images = np.array(images)
    labels = np.array(labels)

    np.savez_compressed(dest_path, images=images, labels=labels)

    return labels_mapping

labels_mapping = create_npz_from_images(r"dataset\ships_gray", r"dataset\train_set.npz")

A custom Dataset object to handle our images.

In [3]:
class CustomDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        image = Image.fromarray(image)
        
        if self.transform:
            image = self.transform(image)
            
        return image, torch.as_tensor(label, dtype=torch.long)


We define our VanillaNet-7 model, the largest model of the paper that fits the depth limit of the competition.

In [ ]:
# The special activation function that adds a lot of non-linearity to the model
# For each channel and each input, the output is a weighted sum of the input and its neighbourhood, after adding a bias and applying a standard activation function (ReLU here)
# The architecture saves computation by mixing channels with 1x1 convolutions and spatial neighbours with the SIAF, instead of doing both at once like a regular convolution.
class SeriesInformedActivation(nn.Module):
    def __init__(self, channels, sia_n=3):
        super(SeriesInformedActivation, self).__init__()
        self.series_informed_bias = nn.Parameter(torch.zeros(1, channels, 1, 1))
    
        # The definition of the SIAF translates directly into a depthwise convolution without bias
        self.series_informed_conv = nn.Conv2d(in_channels=channels,
                                                    out_channels=channels,
                                                    kernel_size=2*sia_n+1,
                                                    padding='same',
                                                    groups=channels,
                                                    bias=False)

    def forward(self, x):
        return self.series_informed_conv(torch.relu(x + self.series_informed_bias))


# Basic block of the architecture
class VanillaNetConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, sia_n=3):
        super(VanillaNetConvBlock, self).__init__()

        self.first_conv = nn.Conv2d(in_channels=in_channels,
                                        out_channels=out_channels,
                                        kernel_size=1)
        self.first_batch_norm = nn.BatchNorm2d(out_channels)
        self.second_conv = nn.Conv2d(in_channels=out_channels,
                                        out_channels=out_channels,
                                        kernel_size=1)
        self.second_batch_norm = nn.BatchNorm2d(out_channels)
        self.series_informed_activation = SeriesInformedActivation(out_channels)

    def internal_activation_1d(self, x, lmbd):
        return (1 - lmbd) * torch.relu(x) + lmbd * x

    def forward(self, x, lmbd):
        # This whole part can be turned into a single convolution at inference
        # ---
        out = self.first_conv(x)
        out = self.first_batch_norm(out)
        out = self.internal_activation_1d(out, lmbd)
        out = self.second_conv(out)
        out = self.second_batch_norm(out)
        # ---
        out = self.series_informed_activation(out)
        return out

# VanillaNet7 adapted to our 192x128 grayscale (single-channel) images
class VanillaNet7(nn.Module):
    def __init__(self, num_classes):
        super(VanillaNet7, self).__init__()

        self.stem = nn.Conv2d(in_channels=1, out_channels=512, kernel_size=4, stride=4) # in_channels=1 because the images are grayscale; this is the only architectural difference with the paper
        self.stem_series_informed_activation = SeriesInformedActivation(512)
        self.stem_batch_norm = nn.BatchNorm2d(512)

        self.block_1_1 = VanillaNetConvBlock(in_channels=512, out_channels=1024)
        self.block_1_2 = VanillaNetConvBlock(in_channels=1024, out_channels=1024)
        self.max_pooling_1 = nn.MaxPool2d(kernel_size=2)

        self.block_2 = VanillaNetConvBlock(in_channels=1024, out_channels=2048)
        self.max_pooling_2 = nn.MaxPool2d(kernel_size=2)

        self.block_3 = VanillaNetConvBlock(in_channels=2048, out_channels=4096)
        self.max_pooling_3 = nn.MaxPool2d(kernel_size=2)

        # We managed to use the paper's channel counts; this gained ~1% on the competition score compared with the same model with every channel count halved
        # The downside is that training time grows roughly linearly with the number of channels
        self.block_4 = VanillaNetConvBlock(in_channels=4096, out_channels=4096)

        # AdaptiveAvgPool2d computes the pooling size for us
        self.classifier_avg_pooling = nn.AdaptiveAvgPool2d(1)
        self.classifier_dropout = nn.Dropout(p=0.05)
        self.classifier_conv = nn.Conv2d(in_channels=4096, out_channels=num_classes, kernel_size=1)

        # Weight initialisation as in the paper
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.trunc_normal_(m.weight, std=0.2)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x, lmbd):
        out = self.stem(x)
        out = self.stem_batch_norm(out)
        out = self.stem_series_informed_activation(out)

        out = self.block_1_1(out, lmbd)
        out = self.block_1_2(out, lmbd)
        out = self.max_pooling_1(out)

        out = self.block_2(out, lmbd)
        out = self.max_pooling_2(out)

        out = self.block_3(out, lmbd)
        out = self.max_pooling_3(out)

        out = self.block_4(out, lmbd)

        out = self.classifier_avg_pooling(out)
        out = self.classifier_dropout(out)
        out = self.classifier_conv(out)
        out = torch.flatten(out, 1)
        return out


Now let's train the model.

We first trained with 20% of the training set held out for validation, to see how the model behaves; the hyperparameters come from this run. The model reached almost its best validation accuracy after 240 epochs, and the remaining epochs up to 300 mostly brought the last tenths of a point. We then trained again on 100% of the training data, without a validation set, since the training loop was known to be well calibrated. Training on more data is one of the most effective ways to gain accuracy.

In [ ]:
MODEL_CLASS = VanillaNet7
TRAIN_ON_FULL_DATASET = True

IMG_HEIGHT = 128
IMG_WIDTH = 192
MEAN = (0.5,)
STD = (0.5,)

# Hyperparameters
NUM_CLASSES = 10
BATCH_SIZE = 64
NUM_EPOCHS = 300
LR = 5e-4
W_DECAY = 0.1
LABEL_SMOOTHING = 0.12
PATIENCE = 14
VAL_EVERY = 10
DEEP_TRAINING_EPOCHS = 200
MIXUP_ALPHA = 0.2
CUTMIX_ALPHA = 1.0
GRAD_ACCUM_STEPS = 2
EMA_DECAY = 0.999

USE_MULTI_GPU = False
CUDA_DEVICE = 0
CHECKPOINT_FOLDER = "checkpoints/run1"
BEST_MODEL_PATH = "best_model.pt"

In [5]:
# Helpers

# Training is long, so we tried to keep the GPU(s) as busy as possible.
# Prefetching avoids idle time and maximises GPU usage.
class CUDAPrefetcher:
    def __init__(self, loader, device):
        self.loader = loader
        self.device = device
        self.stream = torch.cuda.Stream(device=device)

    def __iter__(self):
        first = True
        for next_batch in self.loader:
            with torch.cuda.stream(self.stream):
                next_images = next_batch[0].to(self.device, non_blocking=True)
                next_labels = next_batch[1].to(self.device, non_blocking=True)
            if not first:
                yield images, labels
            first = False
            torch.cuda.current_stream(self.device).wait_stream(self.stream)
            images, labels = next_images, next_labels
        yield images, labels

    def __len__(self):
        return len(self.loader)


def cutmix_bbox(W, H, lam):
    cut_ratio = np.sqrt(1.0 - lam)
    cut_w = max(1, int(W * cut_ratio))
    cut_h = max(1, int(H * cut_ratio))
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    return x1, y1, x2, y2


# Helper that applies cutmix/mixup to the images and, accordingly, to the labels
# This data augmentation blends images, either by pasting a random patch of one image onto another or by overlaying two images with some transparency
def mix_batch(images, labels_smoothed, mixup_alpha, cutmix_alpha):
    if np.random.rand() < 0.5:
        lam = np.random.beta(mixup_alpha, mixup_alpha) if mixup_alpha > 0 else 1.0
        index = torch.randperm(images.size(0), device=images.device)
        mixed = lam * images + (1 - lam) * images[index]
        targets = lam * labels_smoothed + (1 - lam) * labels_smoothed[index]
    else:
        lam = np.random.beta(cutmix_alpha, cutmix_alpha) if cutmix_alpha > 0 else 1.0
        index = torch.randperm(images.size(0), device=images.device)
        mixed = images.clone()
        _, _, H, W = images.shape
        x1, y1, x2, y2 = cutmix_bbox(W, H, lam)
        mixed[:, :, y1:y2, x1:x2] = images[index, :, y1:y2, x1:x2]
        lam_actual = 1.0 - (x2 - x1) * (y2 - y1) / float(W * H)
        targets = lam_actual * labels_smoothed + (1 - lam_actual) * labels_smoothed[index]
    return mixed, targets


# The final model is an EMA of the training model over several epochs, which stabilises it
@torch.no_grad()
def update_ema(ema_model, model, decay):
    for ema_p, p in zip(ema_model.parameters(), model.parameters()):
        ema_p.mul_(decay).add_(p.data, alpha=1 - decay)
    for ema_b, b in zip(ema_model.buffers(), model.buffers()):
        ema_b.copy_(b)


# Evaluation of the EMA model
@torch.no_grad()
def validate_ema(model, val_loader, lmbd):
    torch.cuda.empty_cache()
    model.eval()
    correct = 0
    total = 0
    for images, labels in val_loader:
        with autocast("cuda"):
            logits = model(images, lmbd)
        _, predicted = torch.max(logits, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    return 100.0 * correct / total


def checkpoint_path_for_epoch(checkpoint_folder, epoch):
    os.makedirs(checkpoint_folder, exist_ok=True)
    return os.path.join(checkpoint_folder, f"checkpoint-{epoch + 1:04d}.pt")


def find_latest_checkpoint(checkpoint_folder):
    if not os.path.isdir(checkpoint_folder):
        return None

    checkpoints = sorted(
        f for f in os.listdir(checkpoint_folder)
        if f.startswith("checkpoint-") and f.endswith(".pt")
    )
    if not checkpoints:
        return None

    return os.path.join(checkpoint_folder, checkpoints[-1])


def save_checkpoint(path, epoch, model, ema_model, optimizer, scheduler, scaler, best_val_acc, epochs_no_improve):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "ema_state_dict": ema_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_val_acc": best_val_acc,
        "epochs_no_improve": epochs_no_improve,
    }, path)

In [7]:
# Training function

def train_model(
    model,
    train_loader,
    val_loader=None,
    model_class=MODEL_CLASS,
    num_classes=NUM_CLASSES,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
    label_smoothing=LABEL_SMOOTHING,
    best_model_path=BEST_MODEL_PATH,
    checkpoint_folder=CHECKPOINT_FOLDER,
):
    # Realistically, training on CPU is not an option...
    if torch.cuda.is_available():
        device = torch.device(f"cuda:{CUDA_DEVICE}")
    else:
        device = torch.device("cpu")
    
    # Training model and "final" model, which is the EMA of the first one
    model.to(device)

    ema_model = model_class(num_classes=num_classes).to(device)
    ema_model.load_state_dict(model.state_dict())
    ema_model.eval()

    # Use several GPUs when available
    if USE_MULTI_GPU and torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs via DataParallel")
        train_model_dp = nn.DataParallel(model)
    else:
        print(f"Using single GPU: {torch.cuda.get_device_name(device)}")
        train_model_dp = model

    # Data loaders
    train_loader = CUDAPrefetcher(train_loader, device)
    if val_loader is not None:
        val_loader = CUDAPrefetcher(val_loader, device)

    # BCE as in the paper, since it handles the soft labels produced by cutmix/mixup directly
    criterion = nn.BCEWithLogitsLoss()
    # The paper uses LAMB for ImageNet, which is very large, but AdamW for the smaller COCO; we did the same with our few tens of thousands of images
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=W_DECAY)
    # GradScaler keeps mixed-precision training stable
    scaler = GradScaler("cuda")

    # LR schedule as in the paper: 5 warm-up epochs, then cosine decay
    warmup_scheduler = optim.lr_scheduler.LinearLR(
        optimizer, 
        start_factor=0.01, 
        total_iters=5
    )
    cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs - 5,
        eta_min=1e-6
    )

    scheduler = optim.lr_scheduler.SequentialLR(
        optimizer, 
        schedulers=[warmup_scheduler, cosine_scheduler], 
        milestones=[5]
    )

    best_val_acc = 0.0
    epochs_no_improve = 0
    start_epoch = 0

    os.makedirs(checkpoint_folder, exist_ok=True)

    # Resume from the latest checkpoint if there is one
    latest_checkpoint = find_latest_checkpoint(checkpoint_folder)
    if latest_checkpoint is not None:
        ckpt = torch.load(latest_checkpoint, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        ema_model.load_state_dict(ckpt["ema_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        scaler.load_state_dict(ckpt["scaler_state_dict"])
        best_val_acc = ckpt["best_val_acc"]
        epochs_no_improve = ckpt["epochs_no_improve"]
        start_epoch = ckpt["epoch"] + 1
        print(f"Resumed from {latest_checkpoint} (epoch {start_epoch}), best val acc: {best_val_acc:.2f}%")

    # Epoch loop
    for epoch in range(start_epoch, num_epochs):
        train_model_dp.train()
        running_loss = 0.0
        train_correct = 0
        train_total = 0

        # Lambda controls how the activation between the two convolutions is replaced.
        # With lambda = 1 these activations are the identity
        lmbd = min(epoch / float(DEEP_TRAINING_EPOCHS), 1.0)

        optimizer.zero_grad(set_to_none=True)
        # Batch loop
        for step, (images, labels) in enumerate(train_loader):

            # Label smoothing + mixup/cutmix
            labels_one_hot = torch.nn.functional.one_hot(labels, num_classes=num_classes).float()
            labels_smoothed = labels_one_hot * (1.0 - label_smoothing) + (label_smoothing / num_classes)

            mixed_images, mixed_targets = mix_batch(images, labels_smoothed, MIXUP_ALPHA, CUTMIX_ALPHA)

            with autocast("cuda"):
                # Forward
                outputs = train_model_dp(mixed_images, lmbd)
                # Loss
                loss = criterion(outputs, mixed_targets) / GRAD_ACCUM_STEPS
            # Backward
            scaler.scale(loss).backward()

            # Gradient accumulation, which did not bring a measurable gain
            if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                # Gradient clipping, also for stability
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                # Optimizer step
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                update_ema(ema_model, model, EMA_DECAY)

            # Monitoring metrics
            running_loss += loss.item() * GRAD_ACCUM_STEPS
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
        
        # Scheduler step
        scheduler.step()
        train_acc = 100.0 * train_correct / train_total
        current_lr = optimizer.param_groups[0]['lr']

        should_checkpoint = (epoch + 1) % VAL_EVERY == 0 or epoch == num_epochs - 1
        val_acc = None

        # With a validation set, evaluate the model at each checkpoint
        if val_loader is not None and should_checkpoint:
            val_acc = validate_ema(ema_model, val_loader, lmbd)
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                epochs_no_improve = 0
                torch.save(
                    {
                        "epoch": epoch,
                        "ema_state_dict": ema_model.state_dict(),
                        "val_acc": val_acc,
                    },
                    best_model_path,
                )
            else:
                epochs_no_improve += 1

        if should_checkpoint:
            epoch_checkpoint_path = checkpoint_path_for_epoch(checkpoint_folder, epoch)
            save_checkpoint(epoch_checkpoint_path, epoch, model, ema_model, optimizer, scheduler, scaler, best_val_acc, epochs_no_improve)

        if val_acc is not None:
            print(f"Epoch {epoch+1}/{num_epochs} | Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}% | Val Acc (EMA): {val_acc:.2f}% | LR: {current_lr:.6f} | λ: {lmbd:.2f}")
            if epochs_no_improve >= patience:
                print(f"Early stopping: no improvement for {patience} validation checks ({patience * VAL_EVERY} epochs).")
                break
        else:
            print(f"Epoch {epoch+1}/{num_epochs} | Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}% | LR: {current_lr:.6f} | λ: {lmbd:.2f}")

    # End of training
    if val_loader is not None:
        checkpoint = torch.load(best_model_path, map_location=device)
        ema_model.load_state_dict(checkpoint["ema_state_dict"])
        print(f"Training finished. Best val acc (EMA): {checkpoint['val_acc']:.2f}% — EMA weights restored from {best_model_path}.")
    else:
        torch.save({"ema_state_dict": ema_model.state_dict()}, best_model_path)
        print(f"Training finished ({num_epochs} epochs). EMA weights saved to {best_model_path}.")
    return ema_model



In [ ]:
# Main

# Load the dataset
data = np.load(r"dataset\train_set.npz")
all_images = data['images']
all_labels = data['labels']

val_loader = None

if TRAIN_ON_FULL_DATASET:
    train_images, train_labels = all_images, all_labels
    del data
else:
    indices = np.random.permutation(len(all_images))
    train_split = int(0.8 * len(all_images))
    train_idx = indices[:train_split]
    val_idx = indices[train_split:]

    train_images, train_labels = all_images[train_idx], all_labels[train_idx]
    val_images, val_labels = all_images[val_idx], all_labels[val_idx]
    del data, all_images, all_labels, indices, train_idx, val_idx

# We hit a memory leak once without finding its cause; since then we have been careful with RAM, hence gc.collect and the "del" calls
gc.collect()

# The paper uses RandAugment (heavy data augmentation). It gave worse results here, so we used our own recipe.
train_trans = transforms.Compose([
    transforms.RandomResizedCrop((IMG_HEIGHT, IMG_WIDTH), scale=(0.45, 1.0), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=12, translate=(0.07, 0.07)),
    transforms.RandomAdjustSharpness(sharpness_factor=2, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.2),
])

# Datasets and loaders
train_dataset = CustomDataset(train_images, train_labels, transform=train_trans)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=True,
)

if not TRAIN_ON_FULL_DATASET:
    val_trans = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])
    val_dataset = CustomDataset(val_images, val_labels, transform=val_trans)
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        pin_memory=True,
    )

# Train the model
model = MODEL_CLASS(num_classes=NUM_CLASSES)
ema_model = train_model(model, train_loader, val_loader)

Using single GPU: NVIDIA GeForce RTX 5070 Ti
Epoch 1/300 | Loss: 0.5432 | Train Acc: 14.07% | LR: 0.000104 | λ: 0.00
Epoch 2/300 | Loss: 0.3580 | Train Acc: 23.68% | LR: 0.000203 | λ: 0.01
Epoch 3/300 | Loss: 0.3288 | Train Acc: 27.70% | LR: 0.000302 | λ: 0.01
Epoch 4/300 | Loss: 0.3038 | Train Acc: 30.27% | LR: 0.000401 | λ: 0.01
Epoch 5/300 | Loss: 0.2856 | Train Acc: 31.36% | LR: 0.000500 | λ: 0.02
Epoch 6/300 | Loss: 0.2770 | Train Acc: 35.11% | LR: 0.000500 | λ: 0.03
Epoch 7/300 | Loss: 0.2676 | Train Acc: 36.69% | LR: 0.000500 | λ: 0.03
Epoch 8/300 | Loss: 0.2577 | Train Acc: 37.48% | LR: 0.000500 | λ: 0.04
Epoch 9/300 | Loss: 0.2536 | Train Acc: 41.09% | LR: 0.000500 | λ: 0.04
Epoch 10/300 | Loss: 0.2473 | Train Acc: 40.78% | LR: 0.000500 | λ: 0.04
Epoch 11/300 | Loss: 0.2403 | Train Acc: 40.08% | LR: 0.000499 | λ: 0.05
Epoch 12/300 | Loss: 0.2379 | Train Acc: 41.79% | LR: 0.000499 | λ: 0.06
Epoch 13/300 | Loss: 0.2352 | Train Acc: 43.80% | LR: 0.000499 | λ: 0.06
Epoch 14/300 | 

After 10 to 11 hours on an RTX 5070 Ti and about 4 kWh of electricity, the model is trained. We can now predict the public test set.

In [6]:
CROP_RATIO = 0.9

# Predictions can come from an ensemble of models. We trained a second identical model with another seed, but the ensemble did not score better, so we dropped it.
CHECKPOINT_PATHS = [
    "best_model.pt",
]


# Test-time augmentation: each image is classified from several slightly modified views, and the logits of the views are summed.
# Here, the views are different crops and flips of the same image
def build_tta_transforms(img_h, img_w, crop_ratio, mean, std):
    crop_h = int(img_h * crop_ratio)
    crop_w = int(img_w * crop_ratio)
    pad_h = img_h - crop_h
    pad_w = img_w - crop_w

    offsets = [
        (0, 0),
        (0, pad_w),
        (pad_h, 0),
        (pad_h, pad_w),
        (pad_h // 2, pad_w // 2),
    ]

    tta_transforms = []
    for top, left in offsets:
        for flip in [False, True]:
            ops = [
                transforms.Lambda(lambda img, t=top, l=left, ch=crop_h, cw=crop_w:
                    img.crop((l, t, l + cw, t + ch))),
                transforms.Resize((img_h, img_w), interpolation=transforms.InterpolationMode.BICUBIC),
            ]
            if flip:
                ops.append(transforms.RandomHorizontalFlip(p=1.0))
            ops.extend([
                transforms.ToTensor(),
                transforms.Normalize(mean, std),
            ])
            tta_transforms.append(transforms.Compose(ops))

    return tta_transforms


def load_model_from_checkpoint(checkpoint_path, model_class, num_classes, device):
    model = model_class(num_classes=num_classes).to(device)
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt["ema_state_dict"])
    model.eval()
    return model


@torch.no_grad()
def predict_tta_logits(model, images_np, tta_transforms, device, batch_size=BATCH_SIZE):
    model.eval()
    n = len(images_np)
    all_logits = torch.zeros(n, NUM_CLASSES, device=device)

    for t in tta_transforms:
        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            batch = []
            for i in range(start, end):
                pil_img = Image.fromarray(images_np[i])
                batch.append(t(pil_img))
            batch = torch.stack(batch).to(device)
            with autocast("cuda"):
                all_logits[start:end] += model(batch, lmbd=1.0)

    return all_logits


@torch.no_grad()
def predict_ensemble_tta(images_np, checkpoint_paths, tta_transforms, device, model_class=MODEL_CLASS):
    if isinstance(checkpoint_paths, str):
        checkpoint_paths = [checkpoint_paths]

    n_models = len(checkpoint_paths)
    n = len(images_np)
    ensemble_logits = torch.zeros(n, NUM_CLASSES, device=device)

    for i, path in enumerate(checkpoint_paths):
        model = load_model_from_checkpoint(path, model_class, NUM_CLASSES, device)
        print(f"Loaded checkpoint {i + 1}/{n_models}: {path}")
        ensemble_logits += predict_tta_logits(model, images_np, tta_transforms, device)
        del model
        torch.cuda.empty_cache()

    ensemble_logits /= n_models
    return ensemble_logits.argmax(dim=1).cpu().numpy()

We classify the test set with our model. These predictions scored 0.98295 on the public leaderboard.

In [7]:
device = torch.device(f"cuda:{CUDA_DEVICE}" if torch.cuda.is_available() else "cpu")

X_test = np.load(r"dataset\test_images.npy", allow_pickle=True)
print(f"Test set: {X_test.shape}")

tta_transforms = build_tta_transforms(IMG_HEIGHT, IMG_WIDTH, CROP_RATIO, MEAN, STD)
print(f"Ensemble: {len(CHECKPOINT_PATHS)} model(s), {MODEL_CLASS.__name__}")
print(f"TTA: {len(tta_transforms)} views (5 crops x 2 flips, ratio={CROP_RATIO})")

predictions = predict_ensemble_tta(X_test, CHECKPOINT_PATHS, tta_transforms, device)

df = pd.DataFrame({"Category": predictions})
df.to_csv("reco_nav.csv", index_label="ID")
print(f"Saved predictions to reco_nav.csv ({len(predictions)} samples)")

Test set: (4224, 128, 192)
Ensemble: 1 model(s), VanillaNet7
TTA: 10 views (5 crops x 2 flips, ratio=0.9)
Loaded checkpoint 1/1: best_model.pt
Saved predictions to reco_nav.csv (4224 samples)


We also evaluated the model on the training set. The hyperparameters were found during test runs, so this final run was trained on all the data, without a validation set. Accuracy on the training set does not measure performance, but it shows which classes are misclassified and gives a signal about generalisation.

The model fits the training data completely: with TTA it reaches 100% accuracy. Its 98.3% accuracy on the public test set shows that it still generalises well.

We tried stronger data augmentation and stronger penalties to reduce this overfitting, but those runs quickly collapsed. The training setup seems close to optimal: only a few hundredths of a point are left, which is not worth the time each training run takes.

At inference, the model can be reduced to 7 layers (13 if the depthwise convolutions of the activations are counted).

In [ ]:
# Evaluation on the training set
# This is not a real validation, but it still shows which classes are poorly learned.
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

X_train = globals().get("X_train", train_images)
y_train = globals().get("y_train", train_labels)

train_predictions = predict_ensemble_tta(X_train, CHECKPOINT_PATHS, tta_transforms, device)
train_accuracy = (train_predictions == y_train).mean()
print(f"Train accuracy: {train_accuracy:.4f} ({train_accuracy * 100:.2f}%)")

cm = confusion_matrix(y_train, train_predictions, labels=np.arange(NUM_CLASSES))

fig, ax = plt.subplots(figsize=(10, 10))
display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.arange(NUM_CLASSES))
display.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
ax.set_title("Confusion matrix on the training set")
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
plt.tight_layout()
plt.show()

Loaded checkpoint 1/1: best_model.pt
Train accuracy: 1.0000 (100.00%)
